In [2]:
import pandas as pd

In [7]:
df = pd.read_csv("DataCoSupplyChainDataset.csv", encoding="ISO-8859-1")

In [8]:
df.shape

(180519, 53)

In [11]:
df.isnull()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,False,False,False,False,False,False,False,False,False,False,...,True,False,False,True,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,True,False,False,True,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,True,False,False,True,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,True,False,False,True,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,True,False,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180514,False,False,False,False,False,False,False,False,False,False,...,True,False,False,True,False,False,False,False,False,False
180515,False,False,False,False,False,False,False,False,False,False,...,True,False,False,True,False,False,False,False,False,False
180516,False,False,False,False,False,False,False,False,False,False,...,True,False,False,True,False,False,False,False,False,False
180517,False,False,False,False,False,False,False,False,False,False,...,True,False,False,True,False,False,False,False,False,False


In [12]:
df.isnull().sum()

Type                                  0
Days for shipping (real)              0
Days for shipment (scheduled)         0
Benefit per order                     0
Sales per customer                    0
Delivery Status                       0
Late_delivery_risk                    0
Category Id                           0
Category Name                         0
Customer City                         0
Customer Country                      0
Customer Email                        0
Customer Fname                        0
Customer Id                           0
Customer Lname                        8
Customer Password                     0
Customer Segment                      0
Customer State                        0
Customer Street                       0
Customer Zipcode                      3
Department Id                         0
Department Name                       0
Latitude                              0
Longitude                             0
Market                                0


In [16]:
df = df.drop(columns = ["Order Zipcode","Product Description","Customer Email","Customer Password","Customer Fname","Customer Lname","Customer Street","Product Image"])

In [18]:
df.dtypes

Type                              object
Days for shipping (real)           int64
Days for shipment (scheduled)      int64
Benefit per order                float64
Sales per customer               float64
Delivery Status                   object
Late_delivery_risk                 int64
Category Id                        int64
Category Name                     object
Customer City                     object
Customer Country                  object
Customer Id                        int64
Customer Segment                  object
Customer State                    object
Customer Zipcode                 float64
Department Id                      int64
Department Name                   object
Latitude                         float64
Longitude                        float64
Market                            object
Order City                        object
Order Country                     object
Order Customer Id                  int64
order date (DateOrders)           object
Order Id        

In [19]:
date_columns = [
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]

df[date_columns] = df[date_columns].apply(pd.to_datetime)

In [22]:
df.dtypes

Type                                     object
Days for shipping (real)                  int64
Days for shipment (scheduled)             int64
Benefit per order                       float64
Sales per customer                      float64
Delivery Status                          object
Late_delivery_risk                        int64
Category Id                               int64
Category Name                            object
Customer City                            object
Customer Country                         object
Customer Id                               int64
Customer Segment                         object
Customer State                           object
Customer Zipcode                        float64
Department Id                             int64
Department Name                          object
Latitude                                float64
Longitude                               float64
Market                                   object
Order City                              

In [23]:
df.duplicated().sum()

np.int64(0)

In [24]:
df["Order Item Id"].duplicated().sum()

np.int64(0)

In [25]:
df["Customer Id"].nunique()

20652

In [26]:
dim_customer = df[[
    "Customer Id",
    "Customer City",
    "Customer Country",
    "Customer Segment",
    "Customer State",
    "Customer Zipcode"
]]

In [27]:
dim_customer.shape

(180519, 6)

In [28]:
dim_customer = dim_customer.drop_duplicates(subset=["Customer Id"])

In [29]:
dim_customer.shape

(20652, 6)

In [30]:
dim_product = df[[
    "Product Card Id",
    "Product Name",
    "Product Price",
    "Product Status",
    "Category Name",
    "Category Id"
]]

In [31]:
dim_product.shape

(180519, 6)

In [32]:
dim_product = dim_product.drop_duplicates(subset=["Product Card Id"])

In [33]:
dim_product["Product Card Id"].duplicated().sum()

np.int64(0)

In [34]:
(df["Product Card Id"] == df["Order Item Cardprod Id"]).all()

np.True_

In [35]:
(df["Customer Id"] == df["Order Customer Id"]).all()

np.True_

In [36]:
fact_orders = df[[
    "Order Item Id",
    "Order Id",
    "Customer Id",
    "Order Item Cardprod Id",
    "Sales",
    "Order Item Quantity",
    "Order Item Discount",
    "Order Profit Per Order",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Delivery Status",
    "Late_delivery_risk",
    "order date (DateOrders)",
    "shipping date (DateOrders)",
    "Shipping Mode"
]]

In [37]:
fact_orders.shape

(180519, 15)

In [38]:
fact_orders.rename(
    columns={"Order Item Cardprod Id": "Product Card Id"},
    inplace=True
)

C:\Users\Arpum\AppData\Local\Temp\ipykernel_30636\1540198995.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fact_orders.rename(


In [39]:
fact_orders.columns

Index(['Order Item Id', 'Order Id', 'Customer Id', 'Product Card Id', 'Sales',
       'Order Item Quantity', 'Order Item Discount', 'Order Profit Per Order',
       'Days for shipping (real)', 'Days for shipment (scheduled)',
       'Delivery Status', 'Late_delivery_risk', 'order date (DateOrders)',
       'shipping date (DateOrders)', 'Shipping Mode'],
      dtype='object')

In [40]:
fact_orders["order date (DateOrders)"].min()

Timestamp('2015-01-01 00:00:00')

In [41]:
fact_orders["order date (DateOrders)"].max()

Timestamp('2018-01-31 23:38:00')

In [42]:
start_date = fact_orders["order date (DateOrders)"].min().normalize()
end_date = fact_orders["order date (DateOrders)"].max().normalize()

dim_date = pd.DataFrame({
    "Date": pd.date_range(start=start_date, end=end_date, freq="D")
})

In [43]:
dim_date.head()

,Date
0,2015-01-01
1,2015-01-02
2,2015-01-03
3,2015-01-04
4,2015-01-05


In [44]:
dim_date.shape

(1127, 1)

In [45]:
dim_date["Year"] = dim_date["Date"].dt.year

In [46]:
dim_date.head()

,Date,Year
0,2015-01-01,2015
1,2015-01-02,2015
2,2015-01-03,2015
3,2015-01-04,2015
4,2015-01-05,2015


In [47]:
dim_date["Month"] = dim_date["Date"].dt.month
dim_date["Month Name"] = dim_date["Date"].dt.month_name()
dim_date["Quarter"] = dim_date["Date"].dt.quarter
dim_date["Day"] = dim_date["Date"].dt.day

In [48]:
dim_date.head()

,Date,Year,Month,Month Name,Quarter,Day
0,2015-01-01,2015,1,January,1,1
1,2015-01-02,2015,1,January,1,2
2,2015-01-03,2015,1,January,1,3
3,2015-01-04,2015,1,January,1,4
4,2015-01-05,2015,1,January,1,5


In [49]:
fact_orders["Order Date"] = fact_orders["order date (DateOrders)"].dt.normalize()

C:\Users\Arpum\AppData\Local\Temp\ipykernel_30636\520418277.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fact_orders["Order Date"] = fact_orders["order date (DateOrders)"].dt.normalize()


In [50]:
fact_orders[["order date (DateOrders)", "Order Date"]].head()

,order date (DateOrders),Order Date
0,2018-01-31 22:56:00,2018-01-31
1,2018-01-13 12:27:00,2018-01-13
2,2018-01-13 12:06:00,2018-01-13
3,2018-01-13 11:45:00,2018-01-13
4,2018-01-13 11:24:00,2018-01-13


In [51]:
fact_orders["Customer Id"].isin(dim_customer["Customer Id"])

0         True
1         True
2         True
3         True
4         True
          ... 
180514    True
180515    True
180516    True
180517    True
180518    True
Name: Customer Id, Length: 180519, dtype: bool

In [52]:
fact_orders["Customer Id"].isin(dim_customer["Customer Id"]).all()

np.True_

In [53]:
fact_orders["Product Card Id"].isin(dim_product["Product Card Id"]).all()

np.True_

In [54]:
fact_orders["Order Date"].isin(dim_date["Date"]).all()

np.True_

In [55]:
dim_customer["Customer Id"].is_unique

True

In [56]:
dim_product["Product Card Id"].is_unique

True

In [57]:
fact_orders["Order Item Id"].is_unique

True

In [58]:
fact_orders.columns

Index(['Order Item Id', 'Order Id', 'Customer Id', 'Product Card Id', 'Sales',
       'Order Item Quantity', 'Order Item Discount', 'Order Profit Per Order',
       'Days for shipping (real)', 'Days for shipment (scheduled)',
       'Delivery Status', 'Late_delivery_risk', 'order date (DateOrders)',
       'shipping date (DateOrders)', 'Shipping Mode', 'Order Date'],
      dtype='object')

In [59]:
fact_orders["Shipping Date"] = fact_orders["shipping date (DateOrders)"].dt.normalize()

C:\Users\Arpum\AppData\Local\Temp\ipykernel_30636\3616714851.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fact_orders["Shipping Date"] = fact_orders["shipping date (DateOrders)"].dt.normalize()


In [60]:
fact_orders["Shipping Date"].max()

Timestamp('2018-02-06 00:00:00')

In [62]:
end_date = max(
    fact_orders["Order Date"].max(),
    fact_orders["Shipping Date"].max()
)

In [63]:
end_date

Timestamp('2018-02-06 00:00:00')

In [64]:
dim_date = pd.DataFrame({
    "Date": pd.date_range(start=start_date, end=end_date, freq="D")
})

In [65]:
dim_date["Year"] = dim_date["Date"].dt.year
dim_date["Month"] = dim_date["Date"].dt.month
dim_date["Month Name"] = dim_date["Date"].dt.month_name()
dim_date["Quarter"] = dim_date["Date"].dt.quarter
dim_date["Day"] = dim_date["Date"].dt.day

In [66]:
dim_date.tail()

,Date,Year,Month,Month Name,Quarter,Day
1128,2018-02-02,2018,2,February,1,2
1129,2018-02-03,2018,2,February,1,3
1130,2018-02-04,2018,2,February,1,4
1131,2018-02-05,2018,2,February,1,5
1132,2018-02-06,2018,2,February,1,6


In [67]:
fact_orders["Order Date"].isin(dim_date["Date"]).all()

np.True_

In [68]:
fact_orders["Shipping Date"].isin(dim_date["Date"]).all()

np.True_

In [69]:
fact_orders.isnull().sum()

Order Item Id                    0
Order Id                         0
Customer Id                      0
Product Card Id                  0
Sales                            0
Order Item Quantity              0
Order Item Discount              0
Order Profit Per Order           0
Days for shipping (real)         0
Days for shipment (scheduled)    0
Delivery Status                  0
Late_delivery_risk               0
order date (DateOrders)          0
shipping date (DateOrders)       0
Shipping Mode                    0
Order Date                       0
Shipping Date                    0
dtype: int64

In [70]:
dim_customer.isnull().sum()

Customer Id         0
Customer City       0
Customer Country    0
Customer Segment    0
Customer State      0
Customer Zipcode    3
dtype: int64

In [71]:
dim_product.isnull().sum()

Product Card Id    0
Product Name       0
Product Price      0
Product Status     0
Category Name      0
Category Id        0
dtype: int64

In [72]:
dim_customer[dim_customer["Customer Zipcode"].isnull()]

,Customer Id,Customer City,Customer Country,Customer Segment,Customer State,Customer Zipcode
35704,14577,CA,EE. UU.,Consumer,95758,NaN
46440,17171,CA,EE. UU.,Corporate,95758,NaN
82511,14046,CA,EE. UU.,Consumer,91732,NaN


In [73]:
df[df["Customer Id"].isin([14577, 17171, 14046])][[
    "Customer Id",
    "Customer City",
    "Customer Country",
    "Customer State",
    "Customer Zipcode"
]]

,Customer Id,Customer City,Customer Country,Customer State,Customer Zipcode
35704,14577,CA,EE. UU.,95758,NaN
46440,17171,CA,EE. UU.,95758,NaN
82511,14046,CA,EE. UU.,91732,NaN


In [74]:
dim_date.isnull().sum()

Date          0
Year          0
Month         0
Month Name    0
Quarter       0
Day           0
dtype: int64

In [75]:
fact_orders.to_csv("fact_orders.csv", index=False)
dim_customer.to_csv("dim_customer.csv", index=False)
dim_product.to_csv("dim_product.csv", index=False)
dim_date.to_csv("dim_date.csv", index=False)

In [76]:
import os
print(os.getcwd())

C:\Users\Arpum\Desktop\Project\Supply Chain_Big Data_ Bussiness Intelligence\Dataset


In [77]:
fact_orders["Order Region"] = df["Order Region"]

C:\Users\Arpum\AppData\Local\Temp\ipykernel_30636\877012985.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fact_orders["Order Region"] = df["Order Region"]


In [78]:
fact_orders["Order Region"].value_counts()

Order Region
Central America    28341
Western Europe     27109
South America      14935
Oceania            10148
Northern Europe     9792
Southeast Asia      9539
Southern Europe     9431
Caribbean           8318
West of USA         7993
South Asia          7731
Eastern Asia        7280
East of USA         6915
West Asia           6009
US Center           5887
South of  USA       4045
Eastern Europe      3920
West Africa         3696
North Africa        3232
East Africa         1852
Central Africa      1677
Southern Africa     1157
Canada               959
Central Asia         553
Name: count, dtype: int64

In [80]:
fact_orders.to_csv("fact_orders.csv", index=False)

In [81]:
import os
print(os.path.abspath("fact_orders.csv"))

C:\Users\Arpum\Desktop\Project\Supply Chain_Big Data_ Bussiness Intelligence\Dataset\fact_orders.csv


In [82]:
print(fact_orders.columns)

Index(['Order Item Id', 'Order Id', 'Customer Id', 'Product Card Id', 'Sales',
       'Order Item Quantity', 'Order Item Discount', 'Order Profit Per Order',
       'Days for shipping (real)', 'Days for shipment (scheduled)',
       'Delivery Status', 'Late_delivery_risk', 'order date (DateOrders)',
       'shipping date (DateOrders)', 'Shipping Mode', 'Order Date',
       'Shipping Date', 'Order Region'],
      dtype='object')


In [83]:
check = pd.read_csv("fact_orders.csv")
print(check.columns)

Index(['Order Item Id', 'Order Id', 'Customer Id', 'Product Card Id', 'Sales',
       'Order Item Quantity', 'Order Item Discount', 'Order Profit Per Order',
       'Days for shipping (real)', 'Days for shipment (scheduled)',
       'Delivery Status', 'Late_delivery_risk', 'order date (DateOrders)',
       'shipping date (DateOrders)', 'Shipping Mode', 'Order Date',
       'Shipping Date', 'Order Region'],
      dtype='object')


In [84]:
check = pd.read_csv("fact_orders.csv")

print("Number of columns:", len(check.columns))
print(check.columns.tolist())

Number of columns: 18
['Order Item Id', 'Order Id', 'Customer Id', 'Product Card Id', 'Sales', 'Order Item Quantity', 'Order Item Discount', 'Order Profit Per Order', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Delivery Status', 'Late_delivery_risk', 'order date (DateOrders)', 'shipping date (DateOrders)', 'Shipping Mode', 'Order Date', 'Shipping Date', 'Order Region']


In [85]:
with open("fact_orders.csv", "r", encoding="utf-8") as f:
    print(f.readline())

Order Item Id,Order Id,Customer Id,Product Card Id,Sales,Order Item Quantity,Order Item Discount,Order Profit Per Order,Days for shipping (real),Days for shipment (scheduled),Delivery Status,Late_delivery_risk,order date (DateOrders),shipping date (DateOrders),Shipping Mode,Order Date,Shipping Date,Order Region

